# Momentum, honestly backtested

_A 12-1 cross-sectional book with the look-ahead gaps closed._

**pyportfolios.com** · [/research/cross-sectional-momentum](https://pyportfolios.com/research/cross-sectional-momentum)

> Runs on numpy + pandas + scipy + matplotlib. Synthetic, seeded data — illustrative, not investment advice.

## Synthetic monthly panel with a little embedded momentum

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(3)
T, N = 240, 60                                   # 20y monthly, 60 names
shock = rng.normal(0, 0.06, size=(T, N))
mret = pd.DataFrame(shock)
# inject mild persistence so winners keep winning (then we must NOT cheat to see it)
for t in range(12, T):
    mret.iloc[t] += 0.05 * mret.iloc[t-12:t-1].mean()

## Signal, the single explicit lag, costs

In [ ]:
mom = (1 + mret).rolling(11).apply(np.prod, raw=True) - 1     # info through month t
ranks = mom.rank(axis=1, pct=True)
longs  = (ranks >= 0.9).astype(float)
shorts = (ranks <= 0.1).astype(float)
w = longs.div(longs.sum(axis=1), axis=0) - shorts.div(shorts.sum(axis=1), axis=0)

port = (w.shift(1) * mret).sum(axis=1)            # weights at t earn returns at t+1
turnover = w.diff().abs().sum(axis=1) / 2
net = port - (turnover * 0.0010).shift(1)         # 10 bps round-trip

def sharpe(r): return np.sqrt(12) * r.mean() / r.std()
print(f"gross Sharpe {sharpe(port.dropna()):.2f}")
print(f"  net Sharpe {sharpe(net.dropna()):.2f}")
print(f"avg monthly turnover {turnover.mean():.0%}")

## The decile spread (the momentum signature)

In [ ]:
import matplotlib.pyplot as plt
dec = pd.qcut(mom.stack(), 10, labels=False)
fwd = mret.shift(-1).stack()
avg = fwd.groupby(dec).mean()
plt.figure(figsize=(7, 3.2))
plt.bar(range(1, 11), avg.values * 100,
        color=["#4a4a42"]*7 + ["#0a8a8a"]*3)
plt.title("Average next-month return by momentum decile")
plt.xlabel("decile (1 = losers, 10 = winners)"); plt.ylabel("% / month")
plt.tight_layout(); plt.show()